# B13 Paper Figure Assembly

This notebook uses the same plotting functions as `tools/case_study/assemble_paper_figures.py`, but exposes them one figure at a time for interactive editing in Jupyter.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "case_study" / "assemble_paper_figures.py").exists():
            return candidate
    raise FileNotFoundError("could not locate repo root from the current working directory")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.case_study import assemble_paper_figures as apf


In [ ]:
run_id = "router_lora_case_v1"
config_path = None
figures_dir_override = None
storyline_path_override = None

# For the plateau-style Fig. 4, make sure the run-scoped B10 sweep here includes
# num_loras=1..8 and cache_budget=8192 or larger.
sweeps_root = REPO_ROOT / "artifacts" / "case_study" / run_id / "sweeps"


In [ ]:
paths = apf.resolve_b13_paths(
    config_path=config_path,
    run_id=run_id,
    figures_dir=str(figures_dir_override) if figures_dir_override else None,
    sweeps_dir=str(sweeps_root),
    storyline_path=str(storyline_path_override) if storyline_path_override else None,
)

apf.ensure_paths_exist(
    [
        paths.popularity_path,
        paths.reuse_path,
        paths.cache_curve_path,
        paths.per_request_miss_path,
        paths.tail_breakdown_path,
        paths.num_loras_path,
    ]
)

print(f"run_id: {paths.run_id}")
print(f"sweeps_root: {paths.sweeps_root}")
print(f"figures_dir: {paths.figures_dir}")
print(f"storyline_path: {paths.storyline_path}")


In [ ]:
def preview_and_save(plot_fn, *plot_args):
    fig, record = plot_fn(*plot_args)
    display(fig)
    apf.save_figure(fig, record.output_path)
    print(f"saved: {record.output_path}")
    print(f"claim: {record.claim}")
    print(f"caption: {record.caption}")
    return record


## Draw Figures

Run each cell after editing the plotting code or the inputs you want to inspect.


In [ ]:
fig1_record = preview_and_save(
    apf.plot_fig1_popularity_rank,
    paths.popularity_path,
    paths.figures_dir / "fig1_popularity_rank.pdf",
)


In [ ]:
fig2_record = preview_and_save(
    apf.plot_fig2_reuse_distance,
    paths.reuse_path,
    paths.figures_dir / "fig2_reuse_distance.pdf",
)


In [ ]:
fig3_record = preview_and_save(
    apf.plot_fig3_cache_curve,
    paths.cache_curve_path,
    paths.figures_dir / "fig3_cache_curve.pdf",
)


In [ ]:
fig4_record = preview_and_save(
    apf.plot_fig4_num_loras_vs_p99,
    paths.num_loras_path,
    paths.figures_dir / "fig4_num_loras_vs_p99.pdf",
)


In [ ]:
fig5_record = preview_and_save(
    apf.plot_fig5_tail_breakdown,
    paths.cache_curve_path,
    paths.per_request_miss_path,
    paths.tail_breakdown_path,
    paths.figures_dir / "fig5_tail_breakdown.pdf",
)


In [ ]:
figure_records = [
    fig1_record,
    fig2_record,
    fig3_record,
    fig4_record,
    fig5_record,
]

manifest_path = paths.figures_dir / "figure_manifest.md"
apf.write_manifest(manifest_path, paths.run_id, figure_records)
apf.write_storyline(paths.storyline_path, paths.run_id, figure_records)

print(f"manifest: {manifest_path}")
print(f"storyline: {paths.storyline_path}")


## Full Batch Rebuild

If you do not need to step through figures individually, this cell regenerates all five PDFs, the manifest, and the storyline in one call.


In [ ]:
records, manifest_path, storyline_path = apf.assemble_paper_figures(
    config_path=config_path,
    run_id=run_id,
    figures_dir=str(figures_dir_override) if figures_dir_override else None,
    sweeps_dir=str(sweeps_root),
    storyline_path=str(storyline_path_override) if storyline_path_override else None,
)

print(f"figures: {len(records)}")
print(f"manifest: {manifest_path}")
print(f"storyline: {storyline_path}")
